# Do the current model and Sleeper, when they agree against ADP, pick the right side?

**Descriptive post-hoc research, requested 2026-08-02. NOT pre-registered. NOT live-validated.**
No threshold was chosen after seeing results — the grid `{0, >5, >7.5, >10}` was fixed by the brief,
and every cell at every threshold is reported.

**Question.** When the shipped `fantasy/projections` season-total model and Sleeper's preseason
projection *both* rank a player materially above (or below) his ADP, does he finish on the side they
predicted?

**Signals** (positive = the source ranks the player above his draft price):

    model_gap   = adp_rank - model_rank
    sleeper_gap = adp_rank - sleeper_rank
    actual_gap  = adp_rank - actual_rank
    consensus_score = sign(model_gap) * min(|model_gap|, |sleeper_gap|)

They agree at threshold `t` iff `sign(model_gap)==sign(sleeper_gap)` and **both** `|gap| > t`
(strict). Ranks are integers, so `>7.5` means **at least 8 spots**. A directional hit needs
`sign(actual_gap)==sign(consensus_score)` and `actual_gap != 0`; an actual tie counts as a **miss**
in the primary rate, with a tie-excluded rate reported alongside.

**Inputs.** The saved walk-forward predictions of the current models only — `pred` raw, never the
retired seasonal model, the old blended board, the 2026 fitted values, or the analyst overlay.
Every walk-forward row for season Y was trained on seasons `< Y` (`build_rb_projection.walk_forward`
asserts `(tr.season < Y).all()`); WR/TE/QB import that same engine.

**Exclusions.** QB rookie rows (the arm was held from the shipped surface). Season 2020 (the stored
Sleeper artifact is provenance-contaminated). Availability is *not* filtered out — both systems
forecast season totals, so injuries belong in the target.

**Two populations, reported side by side.** `all_adp` is every walk-forward row carrying an ADP.
`drafted_top180` restricts to `adp_overall_rank <= 180`, the repo's `phase0_benchmark.POOL_SIZE`
draftable universe. The split matters enormously — see the report.

In [ ]:
"""FULL PIPELINE — descriptive post-hoc study, 2026-08-02.
Developed here, then copied cell-by-cell into the research notebook."""
import hashlib, json, sys, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

REPO = Path(r"c:/Users/josep/Desktop/random_stuff/cowork_OS/JoSchoAnalytics")
RES = REPO / "fantasy" / "projections" / "results"
SEAS_CSV = REPO / "fantasy" / "seasonal_projections" / "season_dataset_2014_2025.csv"
OUT = REPO / "fantasy" / "projections" / "research" / "adp_consensus_agreement_2026-08-02"
OUT.mkdir(parents=True, exist_ok=True)

WF_FILES = {"RB": RES / "walkforward_predictions.csv", "WR": RES / "wr_walkforward_predictions.csv",
            "TE": RES / "te_walkforward_predictions.csv", "QB": RES / "qb_walkforward_predictions.csv"}
TEST_SEASONS = [2021, 2022, 2023, 2024, 2025]
THRESHOLDS = [0.0, 5.0, 7.5, 10.0]
PANELS = {"season_2021": [2021], "season_2022": [2022], "season_2023": [2023],
          "season_2024": [2024], "season_2025": [2025],
          "pooled_2024_2025": [2024, 2025], "pooled_2023_2025": [2023, 2024, 2025],
          "pooled_2021_2025": TEST_SEASONS}
POPULATIONS = {"all_adp": None, "drafted_top180": 180}   # repo phase0 convention: top-180 = draftable
SEED, N_PERM, N_BOOT = 20260802, 10_000, 10_000
RUN_TS = datetime.now(timezone.utc).isoformat()


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


INPUTS = {str(p.relative_to(REPO)): {"sha256": sha256(p), "bytes": p.stat().st_size}
          for p in list(WF_FILES.values()) + [SEAS_CSV]}

## 1. Load, join, integrity

Hard-fails if a walk-forward key duplicates within a position, a row misses the `(season, player_id)`
join, a joined position disagrees with the file's position, a test season falls outside 2021-2025, or
a QB rookie row survives the eligibility filter.

The stored `adp_pos_rank` is checked against a reconstruction and **not used** — see the diagnostic.

In [ ]:
# ------------------------------------------------------------------ load / join / integrity
diag = {}
frames = []
for pos, path in WF_FILES.items():
    d = pd.read_csv(path)
    assert not d.duplicated(["season", "player_id"]).any(), f"duplicate walk-forward key in {path.name}"
    assert set(d["season"]) <= set(TEST_SEASONS), f"test season outside 2021-2025 in {path.name}"
    d["file_position"] = pos
    frames.append(d)
wf = pd.concat(frames, ignore_index=True)
assert not wf.duplicated(["season", "player_id", "file_position"]).any()
diag["walkforward_rows"] = {k: int((wf.file_position == k).sum()) for k in WF_FILES}

sd = pd.read_csv(SEAS_CSV)
assert not sd.duplicated(["season", "player_id"]).any()
keep = ["season", "player_id", "position", "team", "adp_half_ppr", "adp_pos_rank",
        "adp_overall_rank", "sleeper_pts_half_ppr", "is_rookie"]
m = wf.merge(sd[keep], on=["season", "player_id"], how="left", validate="one_to_one")
assert m["position"].notna().all(), "a walk-forward row failed the (season, player_id) join"
assert (m["position"] == m["file_position"]).all(), "joined position disagrees with the file's position"
m = m.rename(columns={"position": "pos"}).drop(columns=["file_position"])
m["group"] = np.where(m["grp"] == "rook", "rookie", "veteran")

diag["join"] = {"wf_rows": len(wf), "joined": int(m["pos"].notna().sum()), "unjoined": 0,
                "position_mismatch": 0,
                "sleeper_in_wf_not_in_dataset": int((m.sleeper.notna() & m.sleeper_pts_half_ppr.isna()).sum()),
                "sleeper_in_dataset_not_in_wf": int((m.sleeper.isna() & m.sleeper_pts_half_ppr.notna()).sum()),
                "sleeper_value_max_abs_diff": float(
                    (m.dropna(subset=["sleeper", "sleeper_pts_half_ppr"]).sleeper
                     - m.dropna(subset=["sleeper", "sleeper_pts_half_ppr"]).sleeper_pts_half_ppr).abs().max())}

qbr = m["pos"].eq("QB") & m["grp"].eq("rook")
diag["qb_rookie_rows_dropped"] = int(qbr.sum())
e_all = m[~qbr].copy()
assert not (e_all["pos"].eq("QB") & e_all["grp"].eq("rook")).any(), "QB rookie row survived the filter"
e_all = e_all[e_all.adp_half_ppr.notna() & e_all.pred.notna() & e_all.y.notna()].copy()
diag["eligible_rows_all_adp"] = len(e_all)

# stored adp_pos_rank vs reconstruction — explained, not used
chk = e_all.copy()
chk["recon_model_pop"] = chk.groupby(["season", "pos"])["adp_half_ppr"].rank(method="min")
full_u = sd[sd.adp_half_ppr.notna() & sd.season.isin(TEST_SEASONS)].copy()
full_u["recon_full_u"] = full_u.groupby(["season", "position"])["adp_half_ppr"].rank(method="min")
diag["adp_pos_rank_check"] = {
    "stored_equals_recon_within_model_population": int((chk.recon_model_pop == chk.adp_pos_rank).sum()),
    "of": len(chk),
    "stored_equals_recon_within_full_dataset_adp_universe": int((full_u.recon_full_u == full_u.adp_pos_rank).mean() * len(full_u)),
    "full_universe_rows": len(full_u),
    "explanation": ("adp_pos_rank is joined verbatim from the external ADP source CSV in "
                    "build_season_dataset.py and is ranked over THAT source's universe, which is a "
                    "superset of both the season dataset and the model's walk-forward population. It "
                    "is therefore not reconstructible here and is NOT used; every rank in this study "
                    "is rebuilt inside the stated population.")}

## 2. Ranks and gaps

**Universe A** (production-board analogue): rank all four quantities over the ADP-bearing model
population; a row with no Sleeper projection keeps a missing Sleeper rank and drops out of signal
evaluation but stays in the rank denominators.

**Universe B** (required sensitivity): restrict to rows complete on all four quantities *first*, then
re-rank everything inside that identical common universe.

Ranks are always taken **within (season, position)** — never across positions.

In [ ]:
# ------------------------------------------------------------------ ranks / gaps
def build_ranks(e, universe):
    d = e.copy()
    if universe == "B":
        d = d[d["sleeper"].notna()].copy()
    g = d.groupby(["season", "pos"])
    d["adp_rank"] = g["adp_half_ppr"].rank(method="min", ascending=True)
    d["model_rank"] = g["pred"].rank(method="min", ascending=False)
    d["sleeper_rank"] = g["sleeper"].rank(method="min", ascending=False)
    d["actual_rank"] = g["y"].rank(method="min", ascending=False)
    d["universe"] = universe
    d["model_gap"] = d.adp_rank - d.model_rank
    d["sleeper_gap"] = d.adp_rank - d.sleeper_rank
    d["actual_gap"] = d.adp_rank - d.actual_rank
    d["complete"] = d[["adp_rank", "model_rank", "sleeper_rank", "actual_rank"]].notna().all(axis=1)
    d["agree_dir"] = np.sign(d.model_gap) == np.sign(d.sleeper_gap)
    d["consensus_score"] = np.sign(d.model_gap) * np.minimum(d.model_gap.abs(), d.sleeper_gap.abs())
    for t in THRESHOLDS:
        tag = str(t).replace(".", "p")
        d[f"elig_t{tag}"] = (d.complete & d.agree_dir & (d.model_gap.abs() > t)
                             & (d.sleeper_gap.abs() > t) & (d.consensus_score != 0))
    d["direction"] = np.where(d.consensus_score > 0, "buy",
                              np.where(d.consensus_score < 0, "fade", "none"))
    d["outcome"] = np.where(d.actual_gap == 0, "tie",
                            np.where(np.sign(d.actual_gap) == np.sign(d.consensus_score), "hit", "miss"))
    return d


def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (np.nan, np.nan)
    p, den = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))


def cell(sub, fields):
    n = len(sub)
    hits = int((sub.outcome == "hit").sum()); misses = int((sub.outcome == "miss").sum())
    ties = int((sub.outcome == "tie").sum())
    lo, hi = wilson(hits, n)
    nz = sub[sub.outcome != "tie"]
    rho = (spearmanr(sub.consensus_score, sub.actual_gap).statistic
           if n >= 3 and sub.consensus_score.nunique() > 1 and sub.actual_gap.nunique() > 1 else np.nan)
    return {**fields, "n": n, "hits": hits, "misses": misses, "ties": ties,
            "hit_rate": hits / n if n else np.nan, "wilson_lo": lo, "wilson_hi": hi,
            "n_ex_ties": len(nz),
            "hit_rate_ex_ties": (nz.outcome == "hit").mean() if len(nz) else np.nan,
            "mean_actual_gap": sub.actual_gap.mean() if n else np.nan,
            "median_actual_gap": sub.actual_gap.median() if n else np.nan,
            "spearman_consensus_vs_actual_gap": rho,
            "median_adp_overall": sub.adp_half_ppr.median() if n else np.nan,
            "too_small_n_lt_10": n < 10}

## 3. Every panel, threshold, universe, position, season and direction

Cells with `n < 10` are flagged `too_small_n_lt_10` and carry no directional conclusion.
The summary is then reconstructed from the row-level frame as an integrity check.

In [ ]:
# ------------------------------------------------------------------ build everything
rowlevel, summary = [], []
for popname, cap in POPULATIONS.items():
    e = e_all if cap is None else e_all[e_all.adp_overall_rank <= cap]
    for uni in ("A", "B"):
        d = build_ranks(e, uni)
        d["population"] = popname
        rowlevel.append(d)
        for pname, seasons in PANELS.items():
            p = d[d.complete & d.season.isin(seasons)]
            for t in THRESHOLDS:
                tag = str(t).replace(".", "p")
                sub = p[p[f"elig_t{tag}"]]
                base = {"population": popname, "universe": uni, "panel": pname, "threshold": t}
                summary.append(cell(sub, {**base, "split": "all", "split_value": "all",
                                          "panel_complete_n": len(p)}))
                for sv, ss in sub.groupby("direction"):
                    summary.append(cell(ss, {**base, "split": "direction", "split_value": sv,
                                             "panel_complete_n": len(p)}))
                for sv, ss in sub.groupby("pos"):
                    summary.append(cell(ss, {**base, "split": "position", "split_value": sv,
                                             "panel_complete_n": len(p)}))
                for sv, ss in sub.groupby("season"):
                    summary.append(cell(ss, {**base, "split": "season", "split_value": str(sv),
                                             "panel_complete_n": len(p)}))
                for sv, ss in sub.groupby("group"):
                    summary.append(cell(ss, {**base, "split": "group", "split_value": sv,
                                             "panel_complete_n": len(p)}))

ROW = pd.concat(rowlevel, ignore_index=True)
SUM = pd.DataFrame(summary)
print("row-level rows:", len(ROW), " summary rows:", len(SUM))

# reconstruct-the-summary integrity check
recon = []
for (pop, uni, pname, t, split, sv), _ in SUM.groupby(
        ["population", "universe", "panel", "threshold", "split", "split_value"]):
    pass
def reconstruct(SUMROW, ROW):
    bad = 0
    for _, r in SUMROW.iterrows():
        d = ROW[(ROW.population == r.population) & (ROW.universe == r.universe) & ROW.complete
                & ROW.season.isin(PANELS[r.panel])]
        tag = str(r.threshold).replace(".", "p")
        sub = d[d[f"elig_t{tag}"]]
        if r.split == "direction": sub = sub[sub.direction == r.split_value]
        elif r.split == "position": sub = sub[sub.pos == r.split_value]
        elif r.split == "season": sub = sub[sub.season.astype(str) == r.split_value]
        elif r.split == "group": sub = sub[sub.group == r.split_value]
        if len(sub) != r.n or int((sub.outcome == "hit").sum()) != r.hits:
            bad += 1
    return bad
nbad = reconstruct(SUM.sample(400, random_state=1), ROW)
assert nbad == 0, f"{nbad} summary rows could not be reconstructed from the row-level frame"
print("summary reconstruction check: PASS (400 sampled cells)")

for k, v in INPUTS.items():
    assert sha256(REPO / k) == v["sha256"], f"source artifact changed during the run: {k}"
print("source-artifact stability check: PASS")

## 4. Incremental comparisons

The question is incremental confirmation, not agreement versus a coin flip. Three comparators:

1. among **model** calls above the threshold, Sleeper agrees vs. does not;
2. among **Sleeper** calls above the threshold, the model agrees vs. does not;
3. an empirical null permuting `actual_gap` within season-position 10,000 times, holding signals,
   thresholds and cell sizes fixed.

Lifts get a stratified (season-position) bootstrap CI. The two projection systems are **not**
independent — Sleeper is a public projection and our model is trained on overlapping information —
so no independence is claimed anywhere.

In [ ]:
# ------------------------------------------------------------------ incremental comparators
# Both resamplers precompute ONE index/sign matrix per (population, universe, panel) and reuse it
# across thresholds and comparators, so every threshold is tested against the same null draws.

def perm_sign_matrix(d, n_perm=N_PERM, seed=SEED):
    """(n_perm, N) int8 matrix of sign(actual_gap) after permuting actual_gap within season-position."""
    rng = np.random.default_rng(seed)
    sg = np.sign(d.actual_gap.to_numpy(float)).astype(np.int8)
    out = np.empty((n_perm, len(d)), dtype=np.int8)
    for ix in d.groupby(["season", "pos"]).indices.values():
        ix = np.asarray(ix)
        order = np.argsort(rng.random((n_perm, len(ix))), axis=1)
        out[:, ix] = sg[ix][order]
    return out


def boot_index_matrix(d, n_boot=N_BOOT, seed=SEED + 1):
    """(n_boot, N) stratified (season-position) bootstrap index matrix."""
    rng = np.random.default_rng(seed)
    strata = [np.asarray(ix) for ix in d.groupby(["season", "pos"]).indices.values()]
    return np.concatenate([ix[rng.integers(0, len(ix), size=(n_boot, len(ix)))] for ix in strata], axis=1)


def permutation_p(d, PS, mask, dircol):
    k = int(mask.sum())
    if k == 0:
        return dict(observed=np.nan, null_mean=np.nan, null_p95=np.nan, p_value=np.nan, n=0)
    sg = np.sign(d[dircol].to_numpy(float)).astype(np.int8)
    ag = np.sign(d.actual_gap.to_numpy(float)).astype(np.int8)
    midx = np.flatnonzero(mask.to_numpy())
    obs = float((ag[midx] == sg[midx]).mean())
    null = (PS[:, midx] == sg[midx][None, :]).mean(axis=1)
    return dict(observed=obs, null_mean=float(null.mean()), null_p95=float(np.percentile(null, 95)),
                p_value=float(((null >= obs).sum() + 1) / (len(null) + 1)), n=k)


def boot_lift(d, BI, mask_a, mask_b, dir_a, dir_b):
    ma, mb = mask_a.to_numpy(), mask_b.to_numpy()
    ca = ((np.sign(d.actual_gap) == np.sign(d[dir_a])) & (d.actual_gap != 0)).to_numpy()
    cb = ((np.sign(d.actual_gap) == np.sign(d[dir_b])) & (d.actual_gap != 0)).to_numpy()
    if ma.sum() == 0 or mb.sum() == 0:
        return dict(hr_a=np.nan, hr_b=np.nan, lift=np.nan, lo=np.nan, hi=np.nan,
                    n_a=int(ma.sum()), n_b=int(mb.sum()), boot_ok=0)
    hra, hrb = float(ca[ma].mean()), float(cb[mb].mean())
    SA, SB = ma[BI], mb[BI]                      # (n_boot, N) membership after resampling
    CA, CB = ca[BI], cb[BI]
    na, nb = SA.sum(1), SB.sum(1)
    ok = (na > 0) & (nb > 0)
    lifts = ((CA & SA).sum(1)[ok] / na[ok]) - ((CB & SB).sum(1)[ok] / nb[ok])
    return dict(hr_a=hra, hr_b=hrb, lift=hra - hrb,
                lo=float(np.percentile(lifts, 2.5)), hi=float(np.percentile(lifts, 97.5)),
                n_a=int(ma.sum()), n_b=int(mb.sum()), boot_ok=int(ok.sum()))


comparisons = []
for popname in POPULATIONS:
    for uni in ("A", "B"):
        for pname in ("pooled_2024_2025", "pooled_2023_2025", "pooled_2021_2025"):
            d = ROW[(ROW.population == popname) & (ROW.universe == uni) & ROW.complete
                    & ROW.season.isin(PANELS[pname])].reset_index(drop=True)
            PS, BI = perm_sign_matrix(d), boot_index_matrix(d)
            for t in THRESHOLDS:
                tag = str(t).replace(".", "p")
                both = d[f"elig_t{tag}"]
                mcall = (d.model_gap.abs() > t) & (d.model_gap != 0)
                scall = (d.sleeper_gap.abs() > t) & (d.sleeper_gap != 0)
                base = dict(population=popname, universe=uni, panel=pname, threshold=t,
                            panel_complete_n=len(d))
                perm = permutation_p(d, PS, both, "consensus_score")
                l1 = boot_lift(d, BI, mcall & both, mcall & ~both, "model_gap", "model_gap")
                l2 = boot_lift(d, BI, scall & both, scall & ~both, "sleeper_gap", "sleeper_gap")
                comparisons.append({**base, **{f"perm_{k}": v for k, v in perm.items()},
                                    **{f"vs_model_alone_{k}": v for k, v in l1.items()},
                                    **{f"vs_sleeper_alone_{k}": v for k, v in l2.items()}})
CMP = pd.DataFrame(comparisons)
print("\nCOMPARATORS (pooled_2024_2025):")
print(CMP[CMP.panel == "pooled_2024_2025"][
    ["population", "universe", "threshold", "perm_n", "perm_observed", "perm_null_mean", "perm_null_p95",
     "perm_p_value", "vs_model_alone_hr_b", "vs_model_alone_lift", "vs_model_alone_lo", "vs_model_alone_hi",
     "vs_sleeper_alone_hr_b", "vs_sleeper_alone_lift", "vs_sleeper_alone_lo", "vs_sleeper_alone_hi"]
].round(4).to_string(index=False))

## 5. Descriptive logistic — does agreement add anything beyond gap size?

Outcome: the **model's** directional call is correct. Predictors: `agree`, `|model_gap|`,
`|sleeper_gap|`, position, season. Self-contained Newton-Raphson (statsmodels is not installed in
this interpreter); instability or separation is flagged rather than hidden.

This is descriptive. It is not a fitted product and nothing is selected on it.

In [ ]:
# ------------------------------------------------------------------ descriptive logistic
# Self-contained Newton-Raphson logistic (statsmodels is not installed in this interpreter).
def logit_fit(X, y, names, tol=1e-9, maxit=100):
    b = np.zeros(X.shape[1]); sep = False
    for it in range(maxit):
        eta = X @ b
        p = 1.0 / (1.0 + np.exp(-np.clip(eta, -30, 30)))
        W = p * (1 - p)
        if W.max() < 1e-12: break
        H = X.T @ (X * W[:, None]) + 1e-10 * np.eye(X.shape[1])
        g = X.T @ (y - p)
        step = np.linalg.solve(H, g)
        b = b + step
        if np.max(np.abs(step)) < tol: break
    else:
        sep = True
    eta = X @ b
    p = 1.0 / (1.0 + np.exp(-np.clip(eta, -30, 30)))
    W = p * (1 - p)
    cov = np.linalg.pinv(X.T @ (X * W[:, None]) + 1e-10 * np.eye(X.shape[1]))
    se = np.sqrt(np.clip(np.diag(cov), 0, None))
    z = np.divide(b, se, out=np.full_like(b, np.nan), where=se > 0)
    from scipy.stats import norm as _n
    pv = 2 * (1 - _n.cdf(np.abs(z)))
    ll = float(np.sum(y * np.log(np.clip(p, 1e-12, 1)) + (1 - y) * np.log(np.clip(1 - p, 1e-12, 1))))
    pbar = y.mean()
    ll0 = float(np.sum(y * np.log(pbar) + (1 - y) * np.log(1 - pbar)))
    sep = sep or bool(np.max(np.abs(b)) > 10) or bool(np.nanmax(se) > 50)
    return dict(names=names, coef=b.tolist(), se=se.tolist(), z=z.tolist(), p=pv.tolist(),
                n=int(len(y)), pseudo_r2=float(1 - ll / ll0) if ll0 != 0 else np.nan,
                iterations=it + 1, unstable_or_separated=sep)


def design(d):
    cols, names = [np.ones(len(d))], ["intercept"]
    cols.append(d.agree_dir.astype(float).to_numpy()); names.append("agree")
    cols.append(d.model_gap.abs().to_numpy(float)); names.append("abs_model_gap")
    cols.append(d.sleeper_gap.abs().to_numpy(float)); names.append("abs_sleeper_gap")
    for pos in sorted(d.pos.unique())[1:]:
        cols.append((d.pos == pos).astype(float).to_numpy()); names.append(f"pos[{pos}]")
    for s in sorted(d.season.unique())[1:]:
        cols.append((d.season == s).astype(float).to_numpy()); names.append(f"season[{s}]")
    return np.column_stack(cols), names


logit_out = {}
for popname in POPULATIONS:
    d = ROW[(ROW.population == popname) & (ROW.universe == "B") & ROW.complete
            & ROW.season.isin([2024, 2025])].copy()
    d = d[(d.model_gap != 0) & (d.sleeper_gap != 0) & (d.actual_gap != 0)].copy()
    yv = (np.sign(d.actual_gap) == np.sign(d.model_gap)).astype(float).to_numpy()
    X, names = design(d)
    r = logit_fit(X, yv, names)
    r["note"] = ("outcome = the MODEL's directional call is correct; 'agree' = Sleeper points the "
                 "same way. Descriptive only.")
    logit_out[popname] = r
    print(f"\nLOGIT [{popname}] n={r['n']} pseudoR2={r['pseudo_r2']:.4f} "
          f"unstable={r['unstable_or_separated']}")
    print(pd.DataFrame({"term": names, "coef": r["coef"], "se": r["se"],
                        "z": r["z"], "p": r["p"]}).round(4).to_string(index=False))

## 6. Player-level rows and the audit tables

The largest hits and misses, for audit and possible video examples. Read the `all_adp` table and the
`drafted_top180` table side by side — the difference between them is the headline finding.

In [ ]:
# ------------------------------------------------------------------ player-level export
PL_COLS = ["population", "universe", "season", "pos", "player_id", "player", "team", "group",
           "adp_half_ppr", "adp_overall_rank", "adp_pos_rank", "pred", "sleeper", "y",
           "adp_rank", "model_rank", "sleeper_rank", "actual_rank",
           "model_gap", "sleeper_gap", "actual_gap", "agree_dir", "consensus_score", "direction",
           "complete", "elig_t0p0", "elig_t5p0", "elig_t7p5", "elig_t10p0", "outcome", "model"]
PLAYER = (ROW[PL_COLS].rename(columns={"pos": "position", "pred": "model_pred",
                                       "y": "actual_half_ppr", "model": "model_family"})
          .sort_values(["population", "universe", "season", "position", "adp_rank"]))
PLAYER["max_elig_threshold"] = np.select(
    [PLAYER.elig_t10p0, PLAYER.elig_t7p5, PLAYER.elig_t5p0, PLAYER.elig_t0p0],
    [10.0, 7.5, 5.0, 0.0], default=np.nan)

SHOW = ["season", "position", "player", "adp_overall_rank", "adp_rank", "model_rank", "sleeper_rank",
        "actual_rank", "model_gap", "sleeper_gap", "consensus_score", "actual_gap",
        "actual_half_ppr", "outcome", "group"]


def audit_tables(pop):
    p = PLAYER[(PLAYER.population == pop) & (PLAYER.universe == "A")
               & PLAYER.season.isin([2024, 2025]) & PLAYER.complete & PLAYER.elig_t5p0]
    h, mm = p[p.outcome == "hit"], p[p.outcome == "miss"]
    return (h.nlargest(12, "consensus_score"), h.nsmallest(12, "consensus_score"),
            mm.reindex(mm.consensus_score.abs().sort_values(ascending=False).index).head(15))


for pop in POPULATIONS:
    bh, bf, bm = audit_tables(pop)
    print(f"\n\n########## AUDIT — population={pop}, universe A, 2024-2025, t>5 ##########")
    print("\nLARGEST BUY HITS:"); print(bh[SHOW].to_string(index=False))
    print("\nLARGEST FADE HITS:"); print(bf[SHOW].to_string(index=False))
    print("\nLARGEST MISSES:"); print(bm[SHOW].to_string(index=False))

## 7. Secondary: a contemporaneous dated market (Underdog)

The freshness confound (`PREREGISTRATION.md`, *SLEEPER FRESHNESS ASYMMETRY*): the stored Sleeper
projection is a week-1-eve snapshot while Sleeper ADP is a late-frozen summer aggregate with no
timestamp. Part of any positive result may be late news that entered the projection after some of the
ADP sample had already drafted.

As a robustness check the same thresholds are run against the **final dated Underdog best-ball
window** (W10, or W9 for 2024), reconstructed offline from the sha256-manifested dumps `h11` already
stages. Kept strictly separate: Underdog best ball is a different format from Sleeper half-PPR
redraft. Nothing is written back to any research artifact.

In [ ]:
# ------------------------------------------------------------------ Underdog dated-market check
# SECONDARY, kept separate: Underdog best-ball is a different format from Sleeper half-PPR redraft.
# Reconstructed offline from h11's staged, sha256-manifested Underdog dumps; nothing is written back.
ud_block = {}
try:
    sys.path.insert(0, str(REPO / "fantasy" / "seasonal_projections"))
    import h11_freshness_signal as H11
    H11.verify_manifest()
    uframes = []
    for yr in H11.PANEL:
        wadp, counts = H11.load_windows(yr)
        w = wadp[wadp.grp == H11.FINAL_WIN[yr]].copy()
        w["season"] = yr
        uframes.append(w)
    UD = (pd.concat(uframes, ignore_index=True).rename(columns={"pos_n": "position"})
          .drop_duplicates(["season", "nn", "position"]))
    sdn = sd[["season", "player_id", "norm_name"]]
    ud_e = (e_all.merge(sdn, on=["season", "player_id"], how="left")
            .merge(UD[["season", "nn", "position", "ud_adp"]],
                   left_on=["season", "norm_name", "pos"],
                   right_on=["season", "nn", "position"], how="left"))
    ud_e = ud_e[ud_e.ud_adp.notna()].copy()
    ud_e["adp_half_ppr"] = ud_e["ud_adp"]           # dated market replaces the Sleeper-ADP price
    ud_block["matched_rows"] = len(ud_e)
    ud_block["match_rate_all"] = float(len(ud_e) / len(e_all))
    _d180 = e_all[e_all.adp_overall_rank <= 180]
    ud_block["match_rate_drafted_top180"] = float(
        ud_e.adp_overall_rank.le(180).sum() / len(_d180))
    ud_block["matched_rows_drafted_top180"] = int(ud_e.adp_overall_rank.le(180).sum())
    ud_block["drafted_top180_rows"] = int(len(_d180))
    ud_block["final_window"] = {str(k): v for k, v in H11.FINAL_WIN.items()}
    ud_rows = []
    for popname, cap in POPULATIONS.items():
        sub_e = ud_e if cap is None else ud_e[ud_e.adp_overall_rank <= cap]
        for uni in ("A", "B"):
            d = build_ranks(sub_e, uni)
            for pname in ("pooled_2024_2025", "pooled_2023_2025", "pooled_2021_2025"):
                p = d[d.complete & d.season.isin(PANELS[pname])]
                for t in THRESHOLDS:
                    tag = str(t).replace(".", "p")
                    ud_rows.append(cell(p[p[f"elig_t{tag}"]],
                                        {"market": "underdog_final_window", "population": popname,
                                         "universe": uni, "panel": pname, "threshold": t,
                                         "split": "all", "split_value": "all",
                                         "panel_complete_n": len(p)}))
    UDSUM = pd.DataFrame(ud_rows)
    print("\nUNDERDOG dated-market check (pooled_2024_2025):")
    print(UDSUM[UDSUM.panel == "pooled_2024_2025"][
        ["population", "universe", "threshold", "n", "hits", "misses", "ties", "hit_rate",
         "wilson_lo", "wilson_hi", "spearman_consensus_vs_actual_gap", "too_small_n_lt_10"]
    ].round(4).to_string(index=False))
    ud_block["status"] = "reconstructed"
except Exception as ex:
    UDSUM = pd.DataFrame()
    ud_block = {"status": "BLOCKED", "error": f"{type(ex).__name__}: {ex}"}
    print(f"\nUNDERDOG check BLOCKED: {ex}")

## 8. Write outputs

Re-verifies every source artifact's SHA-256 after the run.

In [ ]:
# ------------------------------------------------------------------ write outputs
if len(UDSUM):
    SUM_OUT = pd.concat([SUM.assign(market="sleeper_adp"), UDSUM], ignore_index=True)
else:
    SUM_OUT = SUM.assign(market="sleeper_adp")
SUM_OUT.to_csv(OUT / "threshold_summary.csv", index=False)
PLAYER.to_csv(OUT / "player_season_results.csv", index=False)
CMP.to_csv(OUT / "incremental_comparisons.csv", index=False)

manifest = {
    "study": "current-model + Sleeper agreement against ADP (descriptive, post-hoc)",
    "requested": "2026-08-02", "run_timestamp_utc": RUN_TS,
    "status": "DESCRIPTIVE POST-HOC RESEARCH — not pre-registered, not live-validated",
    "python": sys.version, "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "seed": SEED, "n_permutations": N_PERM, "n_bootstrap": N_BOOT,
    "inputs": INPUTS,
    "definitions": {
        "model_gap": "adp_rank - model_rank", "sleeper_gap": "adp_rank - sleeper_rank",
        "actual_gap": "adp_rank - actual_rank",
        "consensus_score": "sign(model_gap) * min(|model_gap|, |sleeper_gap|)",
        "eligibility": "sign(model_gap)==sign(sleeper_gap) AND |model_gap|>t AND |sleeper_gap|>t",
        "thresholds": THRESHOLDS,
        "threshold_note": "strict >; ranks are integers, so >7.5 means at least 8 spots",
        "hit": "sign(actual_gap)==sign(consensus_score) and actual_gap != 0; an actual tie counts as a miss",
        "universe_A": "board analogue — rank over the ADP-bearing model population; Sleeper rank NaN where Sleeper is missing",
        "universe_B": "common universe — restrict to complete rows first, then rank all four",
        "population_all_adp": "every walk-forward row with an ADP, a prediction and an actual",
        "population_drafted_top180": "adp_overall_rank <= 180 (repo phase0 draftable-universe convention)",
        "outcome": "half-PPR observed season total; no injury or games-played filter",
    },
    "exclusions": {
        "qb_rookie_rows_dropped": diag["qb_rookie_rows_dropped"],
        "reason": "the QB rookie arm was held from the shipped surface",
        "season_2020": "excluded — the stored Sleeper artifact is provenance-contaminated (near-actual)",
    },
    "diagnostics": diag,
    "row_counts": {"row_level_rows": len(ROW), "player_season_results_rows": len(PLAYER),
                   "threshold_summary_rows": len(SUM_OUT), "comparison_rows": len(CMP)},
    "logistic": logit_out,
    "underdog_secondary": ud_block,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

for k, v in INPUTS.items():
    assert sha256(REPO / k) == v["sha256"], f"source artifact changed during the run: {k}"
print(f"\nwrote -> {OUT}")
for f in sorted(OUT.glob("*")):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")